# 03 — Parse XML → Raw Holdings

Parses every cached information table into one tidy DataFrame (`holdings_raw.parquet`), one row per `<infoTable>` entry, with **all** fields: issuer, title, cusip, value, shares, share_type, put_call, investment_discretion, sole/shared/none.

**Key fixes in `src/parser.py`:**
- **Namespace-safe** parsing — SEC info tables use namespaces with varying prefixes (`ns1:`, `n1:`, none). Matching literal tag names returns 0 rows on namespaced files: the other cause of the old "empty holdings" bug. We match on local tag names.
- **Value-units regime** — before the Jan-2023 technical amendment, `value` was in **thousands of dollars**; after, in whole dollars. We normalize both into `value_usd` so history is comparable.
- Malformed rows are **logged and counted**, never silently swallowed.

In [ ]:
# --- setup (run this cell first, every session) ---
import os, sys, pathlib

# Option A: Google Drive (data persists across sessions -- recommended)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: uploaded zip directly to Colab (data lost on disconnect)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# Option C: running locally
PROJECT_ROOT = pathlib.Path(__file__).resolve().parents[1] if "__file__" in dir() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# Install dependencies if needed (uncomment on Colab)
# import subprocess; subprocess.run(["pip", "install", "-q", "pyarrow", "requests", "pandas", "matplotlib", "numpy"])

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Project root:", PROJECT_ROOT)
print("Ready.")

In [ ]:
from src.parser import build_raw_holdings

holdings_raw = build_raw_holdings()
holdings_raw.head(10)

In [ ]:
# Per-quarter row counts — Berkshire typically reports ~40–130 rows/quarter
holdings_raw.groupby("quarter").agg(
    rows=("cusip", "size"),
    issuers=("issuer", "nunique"),
    total_value_usd=("value_usd", "sum"),
)

In [ ]:
# Field completeness audit
holdings_raw.isna().mean().round(3).sort_values(ascending=False).to_frame("share_missing")